# Intro to PandaPower for IDP Project Optimization

In [ ]:
try:
    from pandapower.networks.power_system_test_cases import case_ieee30
    from pandapower.run import runpp, runopp
except: 
    %pip install pandapower
    %pip install numba
    from pandapower.networks.power_system_test_cases import case_ieee30
    from pandapower.run import runpp, runopp

from tools.limits import bus_vm_pu_limits, line_loading_limits
import profileloader as pl

### Import IEEE Case 30:
Case 30 is stored in a variable, "net", which consists of several dataframes. 
You can conceptualize a dataframe as being like a spreadsheet: It has named rows and columns, and data is stored in individual cells for each row/column pair. 
By default, the IEEE 30 net has a dataframe for each of the following grid components
- bus (30)
- load (21)
- generator (5)
- shunt (2)
- ext_grid (1)
- line (34)
- transformer (7)
- cost (6)

Additional components, such as for storage, motors, etc. can also be added to the net

In [2]:
net = case_ieee30()

In [3]:
net.gen

,name,bus,p_mw,vm_pu,sn_mva,min_q_mvar,max_q_mvar,scaling,slack,in_service,slack_weight,type,controllable,max_p_mw,min_p_mw,id_q_capability_characteristic,reactive_capability_curve,curve_style
0,None,1,40.0,1.045,NaN,-50.0,40.0,1.0,False,True,0.0,None,True,140.0,0.0,<NA>,False,None
1,None,4,0.0,1.010,NaN,-40.0,40.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None
2,None,7,0.0,1.010,NaN,-40.0,10.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None
3,None,10,0.0,1.082,NaN,-24.0,6.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None
4,None,12,0.0,1.071,NaN,-24.0,6.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None


In [4]:
net.load.iloc[0:5,]

,name,bus,p_mw,q_mvar,const_z_p_percent,const_z_q_percent,const_i_p_percent,const_i_q_percent,sn_mva,scaling,in_service,type,controllable
0,None,1,21.7,12.7,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
1,None,2,2.4,1.2,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
2,None,3,7.6,1.6,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
3,None,4,94.2,19.0,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False
4,None,6,22.8,10.9,0.0,0.0,0.0,0.0,NaN,1.0,True,None,False


Specific cells are adjustable based on external data

In [5]:
net.line

,name,std_type,from_bus,to_bus,length_km,r_ohm_per_km,x_ohm_per_km,c_nf_per_km,g_us_per_km,max_i_ka,df,parallel,type,in_service,max_loading_percent,geo
0,None,None,0,1,1.0,3.345408,10.018800,803.812844,0.0,99999.0,1.0,1,ol,True,100.0,None
1,None,None,0,2,1.0,7.875648,28.784448,621.128107,0.0,99999.0,1.0,1,ol,True,100.0,None
2,None,None,1,3,1.0,9.931680,30.265488,560.233194,0.0,99999.0,1.0,1,ol,True,100.0,None
3,None,None,2,3,1.0,2.299968,6.603696,127.879316,0.0,99999.0,1.0,1,ol,True,100.0,None
4,None,None,1,4,1.0,8.224128,34.551792,636.351835,0.0,99999.0,1.0,1,ol,True,100.0,None
5,None,None,1,5,1.0,10.123344,30.718512,569.367431,0.0,99999.0,1.0,1,ol,True,100.0,None
6,None,None,3,5,1.0,2.073456,7.213536,137.013553,0.0,99999.0,1.0,1,ol,True,100.0,None
7,None,None,4,6,1.0,8.015040,20.211840,310.564053,0.0,99999.0,1.0,1,ol,True,100.0,None
8,None,None,5,6,1.0,4.652208,14.287680,258.803378,0.0,99999.0,1.0,1,ol,True,100.0,None
9,None,None,5,7,1.0,2.090880,7.318080,137.013553,0.0,99999.0,1.0,1,ol,True,100.0,None


In [6]:
# for example: set the power consumption of load #1 to 5x its starting value
print("Before: " + str(net.load.at[0,'p_mw']))
net.load.at[0,'p_mw'] = 5 * net.load.iloc[0].p_mw
print("After: " + str(net.load.at[0,'p_mw']))

# and: set line loading limit to 50%
print("Before: " + str(net.line.at[10,'max_loading_percent']))
net.line.at[10,'max_loading_percent'] = 0.000115
print("After: " + str(net.line.at[10,'max_loading_percent']))

Before: 21.7
After: 108.5
Before: 100.0
After: 0.000115


### Power flow analysis
Analysis can be run on net using the runpp function. This adds a new set of dataframes to the net, containing result data for each bus, load, generator, etc.

In [7]:
runpp(net)
net.res_bus[0:5]

,vm_pu,va_degree,p_mw,q_mvar
0,1.060000,0.000000,-354.593394,37.955528
1,1.045000,-7.877331,68.500000,-80.990133
2,1.019979,-8.949676,2.400000,1.200000
3,1.011496,-11.048863,7.600000,1.600000
4,1.010000,-16.387836,94.200000,-16.806864


In [8]:
net.res_gen

,p_mw,q_mvar,va_degree,vm_pu
0,40.0,93.690133,-7.877331,1.045
1,0.0,35.806864,-16.387836,1.010
2,0.0,36.706023,-13.752101,1.010
3,0.0,16.161593,-16.021338,1.082
4,0.0,10.616528,-16.774079,1.071


You can use the result dataframes to check if power flow data is within the defined limits for each object

In [9]:
# For example, buses:
bus_vm_pu_limits(net)
line_loading_limits(net)

Bus #10: Result 1.082 V p.u. is greater than maximum of 1.06 V p.u.
Bus #12: Result 1.071 V p.u. is greater than maximum of 1.06 V p.u.
All line loading within limits


For purposes of optimization, grid components such as generators and loads have a boolean property called "controllable". If a component's "controllable" property is true, that means that the component can be changed during the optimization algorithm, to find which values produce the optimal grid conditions. Values where "controllable" is false are those which must be kept fixed.  
By default, all sources are controllable, and all loads are not. 

In [10]:
runopp(net)

gen vm_pu > bus max_vm_pu for gens [3 4]. Setting bus limit for these gens.


Below, you can see that the controllable grids have had their p_mw values changed. Compare the generator results with the generator results from before:

In [11]:
net.res_gen

,p_mw,q_mvar,va_degree,vm_pu
0,3.231484e+01,39.999998,-3.274809,1.045516
1,6.498302e+01,39.999999,-5.675205,1.041489
2,7.358746e+01,9.999999,-2.299940,1.018290
3,6.245585e+01,5.999999,4.635082,1.042781
4,8.669135e-07,-0.549637,-6.801571,1.038537


In [12]:
# proof that bus max/min values are now accounted for
bus_vm_pu_limits(net)
line_loading_limits(net)

All buses within limits
All line loading within limits


To demonstrate optimization working if we set one of the generators to a fixed, non-controllable value: 

In [13]:
net.gen.at[0,'p_mw'] = 20
net.gen.at[0, 'controllable'] = False
net.gen

,name,bus,p_mw,vm_pu,sn_mva,min_q_mvar,max_q_mvar,scaling,slack,in_service,slack_weight,type,controllable,max_p_mw,min_p_mw,id_q_capability_characteristic,reactive_capability_curve,curve_style
0,None,1,20.0,1.045,NaN,-50.0,40.0,1.0,False,True,0.0,None,False,140.0,0.0,<NA>,False,None
1,None,4,0.0,1.010,NaN,-40.0,40.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None
2,None,7,0.0,1.010,NaN,-40.0,10.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None
3,None,10,0.0,1.082,NaN,-24.0,6.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None
4,None,12,0.0,1.071,NaN,-24.0,6.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None


In [14]:
runpp(net)
bus_vm_pu_limits(net)
line_loading_limits(net)

Bus #10: Result 1.082 V p.u. is greater than maximum of 1.06 V p.u.
Bus #12: Result 1.071 V p.u. is greater than maximum of 1.06 V p.u.
All line loading within limits


In [15]:
runopp(net)
bus_vm_pu_limits(net)
line_loading_limits(net)


gen vm_pu > bus max_vm_pu for gens [3 4]. Setting bus limit for these gens.


All buses within limits
All line loading within limits


The resulting optimized components have changed based on this new constraint. 

In [16]:
net.gen


,name,bus,p_mw,vm_pu,sn_mva,min_q_mvar,max_q_mvar,scaling,slack,in_service,slack_weight,type,controllable,max_p_mw,min_p_mw,id_q_capability_characteristic,reactive_capability_curve,curve_style
0,None,1,20.0,1.045,NaN,-50.0,40.0,1.0,False,True,0.0,None,False,140.0,0.0,<NA>,False,None
1,None,4,0.0,1.010,NaN,-40.0,40.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None
2,None,7,0.0,1.010,NaN,-40.0,10.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None
3,None,10,0.0,1.082,NaN,-24.0,6.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None
4,None,12,0.0,1.071,NaN,-24.0,6.0,1.0,False,True,0.0,None,True,100.0,0.0,<NA>,False,None


In [17]:
net.res_line

,p_from_mw,q_from_mvar,p_to_mw,q_to_mvar,pl_mw,ql_mvar,i_from_ka,i_to_ka,i_ka,vm_from_pu,va_from_degree,vm_to_pu,va_to_degree,loading_percent
0,112.387042,-9.433132,-110.221546,10.069094,2.165497,0.635961,0.465372,0.463255,0.465372,1.060000,0.000000,1.045000,-3.408835,0.000465
1,34.359188,9.132155,-33.831773,-11.655430,0.527414,-2.523275,0.146698,0.152145,0.152145,1.060000,0.000000,1.028696,-2.712197,0.000152
2,3.121387,11.588165,-3.019794,-15.204858,0.101593,-3.616693,0.050231,0.066428,0.066428,1.045000,-3.408835,1.020698,-3.283744,0.000066
3,31.431773,10.455430,-31.293718,-10.941060,0.138056,-0.485630,0.140843,0.142059,0.142059,1.028696,-2.712197,1.020698,-3.283744,0.000142
4,15.777406,-5.739321,-15.664649,1.648780,0.112758,-4.090540,0.070270,0.065933,0.070270,1.045000,-3.408835,1.044909,-5.136387,0.000070
5,2.822753,11.382061,-2.722636,-15.068900,0.100117,-3.686840,0.049083,0.065613,0.065613,1.045000,-3.408835,1.020787,-3.257211,0.000066
6,-1.134358,-0.360571,1.134507,-0.576636,0.000148,-0.937207,0.005101,0.005453,0.005453,1.020698,-3.283744,1.020787,-3.257211,0.000005
7,-1.664905,19.351219,1.842522,-21.084404,0.177617,-1.733185,0.081301,0.090492,0.090492,1.044909,-5.136387,1.022980,-4.528259,0.000090
8,24.828744,-11.387705,-24.642522,10.184404,0.186222,-1.203301,0.117042,0.114005,0.117042,1.020787,-3.257211,1.022980,-4.528259,0.000117
9,-30.714344,15.693270,30.853067,-16.142850,0.138723,-0.449580,0.147788,0.149631,0.149631,1.020787,-3.257211,1.017851,-2.438864,0.000150


In [ ]:
pl.datasource_loads(net)

c:\Users\victor\Documents\College\Fachhochschule Oberosterreich\Interdisciplinary_Project\idp-grid-stability\src\datasource.py:38: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0        67.666667
1        67.666667
2        66.000000
3        64.666667
4        65.666667
           ...    
35035    66.666667
35036    66.666667
35037    64.333333
35038    63.666667
35039    65.000000
Name: Hospital, Length: 35040, dtype: float64' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  load_profile.iloc[:,i] = (load_profile.iloc[:,i] / float(allocations))


           Datum  Composting  Recycling   Hospital  School      Spa  \
0  1/1/2019 0:00    0.403333   0.256667  67.666667   2.510  149.076   
1  1/1/2019 0:15    0.406667   0.260000  67.666667   2.780  158.376   
2  1/1/2019 0:30    0.396667   0.253333  66.000000   2.535  158.376   
3  1/1/2019 0:45    0.403333   0.256667  64.666667   2.360  151.248   
4  1/1/2019 1:00    0.403333   0.256667  65.666667   2.310  142.548   

   Wastewater  Composting 2  Composting 3  Composting 4  ...  Recycling 4  \
0       62.25      0.403333      0.403333      0.403333  ...     0.256667   
1       39.15      0.406667      0.406667      0.406667  ...     0.260000   
2       23.10      0.396667      0.396667      0.396667  ...     0.253333   
3       25.70      0.403333      0.403333      0.403333  ...     0.256667   
4       37.45      0.403333      0.403333      0.403333  ...     0.256667   

   Hospital 2  Hospital 3  Hospital 4  School 2  School 3    Spa 2    Spa 3  \
0   67.666667   67.666667   67.